In [ ]:
import matplotlib.pyplot as plt
import xarray as xr

from conus_biomass import dir_info
from conus_biomass.make_figures import maps
from conus_biomass.train_models.train_all_models import load_data

In [ ]:
fia_data = load_data()

In [ ]:
num_fires = fia_data["fires_occurred"].sum(dim="plotid")
num_measured_plots = (fia_data["measyear_2"] > fia_data["year"]).sum(dim="plotid")
fia_frac_plots = (num_fires * 100 / num_measured_plots).where(num_measured_plots > 10000)
fia_frac_plots = fia_frac_plots.where(fia_frac_plots["year"] >= 1990)

In [ ]:
slope = xr.open_zarr(dir_info.dir_processed + "data_on_ref_grid/1000m/aspect.zarr")
crs = slope["spatial_ref"].crs_wkt

In [ ]:
inputs = xr.open_dataset(dir_info.dir_processed + "data_on_ref_grid/1000m/all_variables.nc")
inputs = inputs.rio.write_crs(crs)

In [ ]:
western_states = maps.SHP_WESTERN.to_crs(crs)

In [ ]:
is_forest = xr.open_dataset(
    dir_info.dir_processed + "data_on_ref_grid/1000m/" + "is_forest_tseries.nc"
)
is_forest = is_forest.rio.write_crs(crs)

In [ ]:
is_forest_clipped = is_forest["is_forest"].rio.clip(western_states.geometry)

# Temporary fix because of error in 2008
# is_forest_clipped.loc[dict(year=2008)] = is_forest_clipped.sel(year=2009).values

In [ ]:
years_after_fire_clipped = inputs["years_after_fire"].rio.clip(western_states.geometry)

In [ ]:
burned_forest = (years_after_fire_clipped == 0) * (is_forest_clipped / 100)
recently_burned_forest = (years_after_fire_clipped <= 10) * (is_forest_clipped / 100)

In [ ]:
burned_forest_tseries = burned_forest.sum(dim=["y", "x"])[:-1]
recently_burned_forest_tseries = recently_burned_forest.sum(dim=["y", "x"])[:-1]
forest_area = (is_forest_clipped / 100).sum(dim=["y", "x"])[:-1]


fig, axes = plt.subplots(nrows=2, ncols=2, figsize=(15, 10))

yvals = burned_forest_tseries * 100 / forest_area
plt.subplot(2, 2, 1)
ax = axes[0, 0]
plt.plot(yvals["year"], yvals, "-", color="firebrick", linewidth=3, markersize=15, label="MTBS")
plt.plot(
    yvals["year"],
    yvals.rolling(year=10, center=True).mean(),
    "--",
    color="firebrick",
    linewidth=1,
    label="MTBS (centered 10-year rolling mean)",
)
plt.axhline(y=0, linestyle=":", color="k")
plt.ylabel("Percent")
ax.spines[["right", "top"]].set_visible(False)
ax.set_title("Forest area burned annually")
plt.plot(fia_frac_plots["year"], fia_frac_plots, color="salmon", linewidth=3, label="FIA")
# plt.plot(fia_frac_plots['year'], fia_frac_plots.rolling(year=10,center=True).mean(), '--',color='salmon', linewidth=1, label='FIA (centered 10-year rolling mean)')
plt.legend()

yvals = recently_burned_forest_tseries * 100 / forest_area
plt.subplot(2, 2, 2)
ax = axes[0, 1]
plt.plot(yvals["year"], yvals, ".-", color="firebrick", linewidth=3, markersize=15)
plt.axhline(y=0, linestyle=":", color="k")
plt.ylabel("Percent")
plt.xlim([1999, 2022])
plt.ylim([-0.3, 10])
ax.spines[["right", "top"]].set_visible(False)
ax.set_title("Forest area burned within past 10 years")

yvals = burned_forest_tseries * 100 / 1e6
plt.subplot(2, 2, 3)
ax = axes[1, 0]
plt.plot(yvals["year"], yvals, ".-", color="firebrick", linewidth=3, markersize=15, label="MTBS")
plt.plot(
    yvals["year"], yvals.rolling(year=10, center=True).mean(), "--", color="firebrick", linewidth=1
)
plt.axhline(y=0, linestyle=":", color="k")
plt.ylabel("Million ha")
plt.legend()
ax.spines[["right", "top"]].set_visible(False)

yvals = recently_burned_forest_tseries * 100 / 1e6
plt.subplot(2, 2, 4)
ax = axes[1, 1]
plt.plot(yvals["year"], yvals, ".-", color="firebrick", linewidth=3, markersize=15)
plt.axhline(y=0, linestyle=":", color="k")
plt.xlim([1999, 2022])
plt.ylabel("Million ha")
ax.spines[["right", "top"]].set_visible(False)

Fires are also missing from the MTBS dataset owing to gaps in the Landsat archive (J. Lecker, pers. comm.). Gaps are particularly evident during the Landsat ‘commercialisation’ period, 1985–99, when the acquisition and archiving of images was largely driven by market demand (Tucker et al. 2004). "Sources and implications of bias and uncertainty in a century of US wildfire activity data"